In [0]:
%sql
create table if not exists project_dev.gold.fact_transactions
(
  TransactionID string,
  SRC_TransactionID string,
  EncounterID string,
  FK_PatientID string,
  FK_ProviderID string,
  FK_DeptID string,
  ICDCode string,
  ProcedureCode string,
  VisitType string,
  ServiceDate date,
  PaidDate date,
  Amount double,
  PaidAmount double,
  AmountType string,
  ClaimID string,
  datasource string,
  refreshed_at timestamp
)


In [0]:
%sql truncate table project_dev.gold.fact_transactions

In [0]:
%sql
insert into project_dev.gold.fact_transactions
select 
  t.TransactionID, 
  t.SRC_TransactionID,
  t.EncounterID,
  concat(t.PatientID,'-',t.datasource ) as FK_Patient_ID,
  case when t.datasource='hos-a' then concat('H1-',t.providerID) else concat('H2-',t.providerID ) end as FK_Provider_ID, 
  concat(t.DeptID,'-',t.datasource ) as FK_Dept_ID, 
  t.ICDCode,
  t.ProcedureCode CPT_Code,
  t.VisitType,
  t.ServiceDate, 
  t.PaidDate,
  t.Amount Charge_Amt, 
  t.PaidAmount Paid_Amt, 
  t.AmountType,
  t.ClaimID,
  t.datasource,
  current_timestamp()
  from project_dev.silver.transactions t 
  where t.is_current=true and t.is_quarantined=false

In [0]:
%sql
--Total Charge Amount per provider by department
select
  concat(p.firstname, ' ', p.LastName) Provider_Name,
  dd.Name Dept_Name,
  sum(ft.Amount)
from
  project_dev.gold.fact_transactions ft
  left join project_dev.gold.dim_provider p on p.ProviderID = ft.FK_ProviderID
  left join project_dev.gold.dim_department dd on dd.Dept_Id = p.DeptID
group by
  all 

In [0]:
%sql
--Total Charge Amount per provider by department for each month for year 2024
select
  concat(p.firstname, ' ', p.LastName) Provider_Name,
  dd.Name Dept_Name,
  date_format(servicedate, 'yyyyMM') YYYYMM,
  sum(ft.Amount) Total_Charge_Amt,
  sum(ft.paidamount) Total_Paid_Amt
from
  project_dev.gold.fact_transactions ft
  left join project_dev.gold.dim_provider p on p.ProviderID = ft.FK_ProviderID
  left join project_dev.gold.dim_department dd on dd.Dept_Id = p.DeptID
where
  year(ft.ServiceDate) = 2024
group by
  all
order by
  1,
  3